# The Loop, Made Visible: LangGraphNotebook 1: you wrote the agent loop by hand. Notebooks 2 and 3: two frameworks tookthat loop away from you and hid it, in different shapes.**LangGraph does the opposite.** It hands the loop back, but as a diagram — a graph ofnodes and edges over a shared state object. You declare the cycle explicitly:model → decide → tools → back to model. Then it draws it for you.This is the notebook that answers the question a good student asks after notebook 2:*what do I do when the framework's control flow isn't the control flow I want?***What you need:** the same `.env` with your `OPENROUTER_API_KEY`. Nothing new.

In [ ]:
%pip install --quiet langgraph langchain-openai python-dotenv pytestimport osimport subprocessimport sysfrom pathlib import Pathfrom dotenv import find_dotenv, load_dotenvfrom langchain_core.tools import toolfrom langchain_openai import ChatOpenAIfrom langgraph.graph import END, START, MessagesState, StateGraphfrom langgraph.prebuilt import ToolNode, tools_condition# ":free" is OpenRouter's no-cost, rate-limited endpoint for this model.# Drop the suffix for the paid endpoint if the rate limit gets in your way.MODEL = "nvidia/nemotron-3.5-lightning:free"WORKDIR = Path.cwd()load_dotenv(find_dotenv())llm = ChatOpenAI(    model=MODEL,    api_key=os.environ["OPENROUTER_API_KEY"],    base_url="https://openrouter.ai/api/v1",    max_tokens=1024,)print("model   :", MODEL)print("workdir :", WORKDIR)

## 1. Does the API work?`llm.invoke()` is a plain request — no graph, no tools, no state. One message out, onemessage back.

In [ ]:
response = llm.invoke("Write a haiku about debugging code.")print(response.content)print("\nusage:", response.usage_metadata)

## 2. The taskIdentical to every other notebook in this repo.> **Re-run the `%%writefile buggy.py` cell to put the bug back** and start clean.

In [ ]:
%%writefile buggy.pydef add_reading(reading, log=[]):    """Append a sensor reading to a log and return the log."""    log.append(reading)    return logdef average(readings):    """Return the mean of a list of readings."""    return sum(readings) / len(readings)

In [ ]:
%%writefile test_buggy.pyfrom buggy import add_reading, averagedef test_average():    assert average([2, 4, 6]) == 4def test_logs_are_independent():    first = add_reading(1)    second = add_reading(2)    assert first == [1]    assert second == [2]

In [ ]:
print(subprocess.run(    [sys.executable, "-m", "pytest", "-q"],    capture_output=True, text=True, cwd=WORKDIR,).stdout)

## 3. The toolsSame three functions. Third framework, **third spelling of parameter descriptions**:| | how you describe a parameter ||---|---|| by hand (notebook 1) | you wrote the JSON yourself || Agent Framework | `Annotated[str, "..."]` || CrewAI | `Field(..., description="...")` || LangGraph | a Google-style `Args:` docstring **plus** `parse_docstring=True` |That flag is not optional and it is easy to miss. Without it LangChain dumps your entiredocstring — `Args:` block and all — into the tool description and generates a schema withno parameter descriptions at all. Nothing errors. The agent just works less well.Second time you have seen this exact failure mode in this repo. It is worth noticing thatevery framework has invented its own incompatible way to express one idea, and thatgetting it subtly wrong is silent in all of them.

In [ ]:
@tool(parse_docstring=True)def read_file(path: str) -> str:    """Read the full contents of a file in the working directory.    Args:        path: File to read, e.g. buggy.py    """    return (WORKDIR / path).read_text()@tool(parse_docstring=True)def edit_file(path: str, old: str, new: str) -> str:    """Replace an exact snippet of text in a file.    Args:        path: File to edit, e.g. buggy.py        old: Exact text to replace. Must appear exactly once, whitespace included.        new: Replacement text.    """    p = WORKDIR / path    text = p.read_text()    if text.count(old) == 0:        return "ERROR: 'old' not found in the file. Read it again and match it exactly."    if text.count(old) > 1:        return "ERROR: 'old' appears more than once. Include more surrounding context."    p.write_text(text.replace(old, new))    return f"ok, edited {path}"@tool(parse_docstring=True)def run_tests() -> str:    """Run the pytest suite and return its output."""    result = subprocess.run(        [sys.executable, "-m", "pytest", "-q"],        capture_output=True, text=True, timeout=60, cwd=WORKDIR,    )    return (result.stdout + result.stderr)[-2000:] or "(no output)"TOOLS = [read_file, edit_file, run_tests]# Confirm the descriptions actually reached the schema:print(edit_file.description)print(edit_file.args_schema.model_json_schema()["properties"])

## 4. The graph — this is the part you writeHere is the whole idea. Your loop from notebook 1 had exactly two things in it: *ask themodel*, and *run the tools it asked for*. Those become **nodes**. The `if notmsg.tool_calls: break` became an **edge with a condition on it**.```START ──> model ──(tool calls?)──> tools ──┐                 │                          │                 └──(no)──> END             └──> back to model```Four pieces:- **State.** `MessagesState` is a dict with one key, `messages`, and an annotation saying  new messages get appended rather than replacing the list. That is your `messages` list  from notebook 1, given a type.- **The model node.** A plain function: take the state, call the model, return the new  message. `bind_tools` is what attaches the schemas.- **The tool node.** `ToolNode` does the dispatch you wrote by hand — reads `tool_calls`,  looks each one up, runs it, appends a `ToolMessage`.- **The conditional edge.** `tools_condition` is a supplied function that returns `"tools"`  if the last message has tool calls and `"__end__"` otherwise. It is your `if` statement.Note there is no turn cap here. LangGraph counts **steps**, not turns, via`recursion_limit`, and one turn of your old loop is two steps (model, then tools). Soeight turns is roughly sixteen steps. The default is 25.

In [ ]:
llm_with_tools = llm.bind_tools(TOOLS)def call_model(state: MessagesState):    """The model node: send the conversation, return whatever came back."""    return {"messages": [llm_with_tools.invoke(state["messages"])]}builder = StateGraph(MessagesState)# TODO 1: add the two nodes.#   builder.add_node("model", call_model)#   builder.add_node("tools", ToolNode(TOOLS))# TODO 2: wire the edges.#   from START into "model"#   a conditional edge out of "model" using tools_condition#   a plain edge from "tools" back to "model"# TODO 3: compile it.graph = ...

## 5. Look at itThis is what the other two frameworks could not give you. Your agent is now an objectyou can draw, and the picture is the loop you wrote in notebook 1.

In [ ]:
from IPython.display import Markdown, displaydisplay(Markdown("```mermaid\n" + graph.get_graph().draw_mermaid() + "\n```"))

## 6. Run it`stream()` is where the transcript comes from here — no middleware, no callbacks. Eachchunk is one node finishing, so you are watching the graph traverse itself.

In [ ]:
TASK = (    "The tests in test_buggy.py are failing. Read buggy.py, find the bug, fix it, "    "and run the tests until they pass. When they pass, say in one sentence what the "    "bug was.")CONFIG = {"recursion_limit": 16}      # ~8 turns: each turn is a model step and a tool stepdef show(node, message):    kind = message.__class__.__name__    for call in getattr(message, "tool_calls", None) or []:        print(f"  -> {call['name']}({call['args']})")    text = getattr(message, "content", "")    if text:        label = "  <-" if kind == "ToolMessage" else f"  [{node}]"        print(f"{label} {str(text)[:300]}")for chunk in graph.stream({"messages": [("user", TASK)]}, config=CONFIG,                          stream_mode="updates"):    for node, update in chunk.items():        for message in (update or {}).get("messages", []):            show(node, message)

## 7. Did it actually change the file?

In [ ]:
print(Path("buggy.py").read_text())print(subprocess.run(    [sys.executable, "-m", "pytest", "-q"],    capture_output=True, text=True, cwd=WORKDIR,).stdout)

## 8. What LangGraph is actually forEverything so far has been the same agent. The reason to reach for a graph is whathappens when you want something the straight loop cannot express:- **A node that is not the model.** A validator, a linter, a database write, a human  approval step — anything can be a node, and only some of them are LLM calls.- **Branching that is not "tool or done".** Route to different tools, different models,  or different subgraphs based on the state.- **Checkpointing.** Add a checkpointer and the graph can pause mid-run, persist, and  resume later — which is how real human-in-the-loop approval works, as opposed to a  prompt politely asking the model to wait.- **Loops you control.** Retry a node N times, or cycle between two nodes until a  condition holds.None of that is available to you in `agent.run()` or `crew.kickoff()` without leaving theframework. That is the trade: more ceremony for a simple agent, and far more reach for acomplicated one.### The honest caveatLangGraph also has `create_react_agent`, a one-line prebuilt that does everything thisnotebook built by hand:```pythonfrom langgraph.prebuilt import create_react_agentgraph = create_react_agent(llm, TOOLS)```That is what you would actually write in production for an agent this simple. We builtthe graph explicitly because the graph *is* the lesson — but you should know the shortcutexists, and that reaching for it puts you right back where notebook 2 left you.### Try this1. Set `recursion_limit` to 4 and watch it raise rather than stop politely. Compare with   how your hand-written loop behaved when it hit `max_turns`.2. Add a third node between `tools` and `model` that just prints the state, and re-draw   the graph. Notice that changing the control flow is a one-line edit here.